In [1]:
# this notebook is perform anaylsis of MACE events in the care home hypertension cohort 

In [2]:
library(ggplot2)
library(bigrquery)
library(tidyverse)
library(ggplot2)
library(lubridate)
library(stats)

ProjectId = "yhcr-prd-bradfor-bia-core"

targetdb1 <-'yhcr-prd-bradfor-bia-core.CB_MYSPACE_MK'
targetdb1 <-gsub(' ','',targetdb1)

targetdb2 <- 'yhcr-prd-bradfor-bia-core.CB_FDM_DeathCertificates'
targetdb2 <-gsub(' ','',targetdb2)

targetdb3 <- 'yhcr-prd-bradfor-bia-core.CB_2172'
targetdb3 <- gsub(' ','',targetdb3)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
sql1 <- 
paste(
    'SELECT  *
from ',targetdb3,'.cb_2172_mace 
;',sep="")

mace_tbl <-bq_project_query(ProjectId,sql1)
mace_data <- bq_table_download(mace_tbl)
#filter out any individuals where the time between admission and death is less than 42 days 


In [4]:
mace_data$date.event <- as.Date(mace_data$dod)

mace_data <- mace_data %>% select(-dod)

In [5]:
mace_data <- mace_data %>% arrange(person_id, desc(date.event)) %>% mutate(event_id= rank(date.event
                                                                                        ))
#here is the initial data, the problem is there are some  duplicate events where the data differs by only a couple of days so they are counted as different events. 
#Some cardiovasccular deaths also have  the cardiovascular event (e.g. myocardial evvent, recorded as a seperate event a few days apart). 
#I need to remove these events so that only the cardiovascular death is recorded. 
#This is based on the assumption that where a stroke or MI has been recorded within 30 days of a cardiovascular death, the two are realted and therefore the stroke or MI was a fatal one so should be recorded as a CV death rather than a non fatal MI or stroke. 
mace_data

person_id,event_type,date.event,event_id
<int>,<chr>,<date>,<dbl>
339912,cv death,2017-12-15,72.5
533373,cv death,2017-11-15,65.5
576388,cv death,2019-10-15,180.0
658432,cv death,2019-05-15,156.0
706765,cv death,2019-03-15,140.0
879366,cv death,2019-06-15,162.5
879366,myocardial infarction,2019-06-10,159.0
899131,myocardial infarction,2020-05-26,201.0
899131,cv death,2020-05-15,200.0


In [ ]:
# this identifies myocardial infarction events where another event occurs afterwards(either a duplicate event or death), so these myocardial events can be removed, i will do the same for stroke as well. this ensures only non fatal myocardial infarctions and strokes are captured in the MACE outcomes. fatal myocardial infarctions and stroke would fall into CV death 
mace_data_filtered_mi_cvd <- mace_data %>%
  group_by(person_id) %>% filter(event_type == 'myocardial infarction' & !((date.event - lag(date.event)) >= 30 & (date.event - lag(date.event)) > 0)) %>%
  ungroup() 

mace_data_filtered_mi_cvd

In [ ]:
mace_data

In [ ]:
mace_data <- mace_data %>% arrange(person_id, desc(date.event))


# this identifies stroke events where another event occurs afterwards(either a duplicate event or death), so these myocardial events can be removed, i will do the same for stroke as well. this ensures only non fatal myocardial infarctions and strokes are captured in the MACE outcomes. fatal myocardial infarctions and stroke would fall into CV death 
mace_filtered_stroke_cvd <- mace_data %>%
  group_by(person_id) %>% filter(event_type == 'stroke' & !((date.event - lag(date.event)) >= 30 & (date.event - lag(date.event)) > 0)) %>%
  ungroup() 

mace_filtered_stroke_cvd

In [9]:
#i then subsetted the mace_Data to remove those duplicate stroke and myocardial events identified above

In [10]:
#creat he not in function

'%notin%' <- Negate(`%in%`)

In [11]:
mace_data_filtered <- mace_data[mace_data$event_id %notin% mace_data_filtered_mi_cvd$event_id, ]

In [12]:
mace_data_filtered <- mace_data_filtered[mace_data_filtered$date.event %notin% mace_filtered_stroke_cvd$date.event, ]

In [16]:
mace_data_filtered <- mace_data_filtered %>% arrange(person_id, desc(date.event))

mace_data_filtered


person_id,event_type,date.event,event_id
<int>,<chr>,<date>,<dbl>
339912,cv death,2017-12-15,72.5
533373,cv death,2017-11-15,65.5
576388,cv death,2019-10-15,180.0
658432,cv death,2019-05-15,156.0
706765,cv death,2019-03-15,140.0
879366,cv death,2019-06-15,162.5
899131,myocardial infarction,2020-05-26,201.0
899131,cv death,2020-05-15,200.0
12388259,cv death,2017-02-15,16.5


In [ ]:
# merge the care home cohort table in order to calculate the incidence density

In [28]:
sql1 <- 
paste(
    'SELECT person_id,first_episodestartdate, hypertension
from ',targetdb3,'.cb_2172_hypertension_combined 
order by person_id
;',sep="")

bp_tbl <-bq_project_query(ProjectId,sql1)
ch_cohort <- bq_table_download(bp_tbl)

ch_cohort$person_id <- as.character(ch_cohort$person_id)
print(ch_cohort)

# A tibble: 2,794 × 3
   person_id first_episodestartdate hypertension
   <chr>     <dttm>                 <lgl>       
 1 1271      2019-01-07 00:00:00    TRUE        
 2 2355      2018-01-03 00:00:00    FALSE       
 3 5275      2019-02-13 00:00:00    TRUE        
 4 30746     2017-07-28 00:00:00    TRUE        
 5 76037     2017-11-01 00:00:00    FALSE       
 6 81564     2017-11-28 00:00:00    FALSE       
 7 111197    2018-02-24 00:00:00    TRUE        
 8 118634    2017-03-30 00:00:00    TRUE        
 9 144752    2017-08-03 00:00:00    TRUE        
10 158375    2018-03-26 00:00:00    TRUE        
# ℹ 2,784 more rows


In [39]:
ch_mace_merged <- merge(x =ch_cohort, y = mace_data_filtered, all.x = TRUE)

In [40]:
ch_mace_merged$first_episodestartdate <- as_date(ch_mace_merged$first_episodestartdate)

In [41]:
ch_mace_merged <- ch_mace_merged %>% mutate(exposure_time = (date.event -first_episodestartdate))

In [42]:
 ch_mace_merged_filtered <- ch_mace_merged %>% filter(hypertension == TRUE)

In [43]:
ch_mace_merged_filtered$exposure_time <- ifelse(is.na(ch_mace_merged_filtered$exposure_time) ,365,ch_mace_merged_filtered$exposure_time)

In [46]:
ch_mace_merged_filtered <-ch_mace_merged_filtered %>%  select(-"event_id")

ERROR: [1m[33mError[39m in `select()`:[22m
[33m![39m Can't select columns that don't exist.
[31m✖[39m Column `event_id` doesn't exist.


In [47]:
(sum(ch_mace_merged_filtered$exposure_time < 365)/(ch_mace_merged_filtered$exposure_time %>% sum()/365))*100

[1] 7.487116